In [3]:
# 11.2
from sympy import *

L = 7
x, w, E, I, C1, C2 = symbols('x w E I C1 C2')

# 第一段方程及导数
y1 = (w/(E*I)) * (-3*L**2*x**2/16 + L*x**3/12)
dy1 = diff(y1, x)

# 第二段弯矩 (直接套负弯矩符号约定)
M2 = -w*(L-x)**2/2

# 积分两次
dy2 = integrate(M2/(E*I), x) + C1
y2 = integrate(dy2, x) + C2

# 在 L/2 处的边界连续条件 (相减等于0)
eq1 = dy1.subs(x, L/2) - dy2.subs(x, L/2)
eq2 = y1.subs(x, L/2) - y2.subs(x, L/2)

# 直接解出 C1, C2
ans = solve([eq1, eq2], [C1, C2])

# 代入答案，除以 w/EI 后展开
y2_ans = expand(y2.subs(ans) / (w/(E*I)))

# 提取系数并翻转顺序 (匹配题目的 A0 到 A4)
coeffs = Poly(y2_ans, x).all_coeffs()
coeffs.reverse()

print(y2_ans)

-x**4/24 + 7*x**3/6 - 49*x**2/4 + 7.14583333333333*x - 6.25260416666667


In [4]:
# 11.3
from sympy import *

x, C1, C2, C3, C4 = symbols('x C1 C2 C3 C4')

# 1. 输入参数
L = 5
P = 7
M = 14
EI = (163 * 10**6) * (56 * 10**-6) 

# 2. 第一段弯矩与积分 (注意这里 M 前面改成了减号！)
M1 = P*(L - x) - M
dy1 = integrate(M1/EI, x) + C1
y1 = integrate(dy1, x) + C2

# 3. 第二段弯矩与积分
M2 = 0
dy2 = integrate(M2/EI, x) + C3
y2 = integrate(dy2, x) + C4

# 4. 边界与连续性条件
ans = solve([
    dy1.subs(x, 0), 
    y1.subs(x, 0),
    dy1.subs(x, L) - dy2.subs(x, L),
    y1.subs(x, L) - y2.subs(x, L)
], [C1, C2, C3, C4])

# 5. 代入展开
y1_ans = expand(y1.subs(ans))
y2_ans = expand(y2.subs(ans))

# 提取并打印系数
def print_coeffs(expr, name):
    c = Poly(expr, x).all_coeffs()
    c.reverse()
    c = (c + [0]*5)[:5]
    print(f"=== {name} ===")
    for i in range(5):
        print(f"x^{i} = {float(c[i]):.10f}")

print_coeffs(y1_ans, "y1(x)")
print_coeffs(y2_ans, "y2(x)")

=== y1(x) ===
x^0 = 0.0000000000
x^1 = 0.0000000000
x^2 = 0.0011503067
x^3 = -0.0001278119
x^4 = 0.0000000000
=== y2(x) ===
x^0 = 0.0031952965
x^1 = 0.0019171779
x^2 = 0.0000000000
x^3 = 0.0000000000
x^4 = 0.0000000000


In [ ]:
# 11.4
from sympy import *

# 1. 统一单位输入 (kN, m)
M = 61 
L = 1.795 
delta = 0.003 
E = 147 * 10**6  # GPa 转换为 kPa (kN/m^2)
I = 115.624 * 10**-6  # mm^4 转换为 m^4

EI = E * I
F = symbols('F')

# 2. 叠加法推导自由端挠度
# 弯矩 M 引起的向下挠度 (跨度2L，力偶位于L处)
delta_M = (M * L**2) / (2 * EI) + (M * L / EI) * L 

# 支座反力 F 引起的向上挠度 (作用在全长 2L 的自由端)
delta_F = F * (2*L)**3 / (3 * EI)

eq = delta_M - delta_F - delta

ans = solve(eq, F)
print(f"|F| = {float(ans[0]):.4f} kN")

|F| = 15.8094 kN


In [6]:
# 11.5
from sympy import *

# 1. 统一单位输入 (kN, m)
P = 3
L = 8
H = 2
E = 240 * 10**6      # GPa 转换为 kPa (kN/m^2)
I = 54 * 10**-6      # mm^4 转换为 m^4
A = 220 * 10**-6     # mm^2 转换为 m^2

EI = E * I
EA = E * A

# 2. 静力学平衡计算杆 CD 的受力
# 绕铰链 A 取矩: P * (L/2) - F_C * L = 0
F_C = P / 2

# 3. 叠加法计算
# 状态 1: 假设 C 点不发生位移，简支梁跨中受集中力引起的弯曲挠度
delta_B_bending = (P * L**3) / (48 * EI)

# 状态 2: 杆 CD 弹性拉伸引起的刚体旋转
# 先计算杆 C 的伸长量 delta_C
delta_C = (F_C * H) / EA
# 因为 B 点在梁的中点，由相似三角形可知，B 点随之下降的位移是 C 点的一半
delta_B_rigid = delta_C / 2

# 4. 总挠度求和并转换单位为 mm
delta_B_total = delta_B_bending + delta_B_rigid
ans_mm = delta_B_total * 1000

print(f"|yb| = {ans_mm:.4f} mm")

|yb| = 2.4975 mm
